# 03 - Semantic Mapping, Clustering, and Stability

                This notebook treats UMAP as an exploratory visualization and performs clustering in higher-dimensional embedding/PCA space or on a kNN graph. It writes all cluster assignments and stability diagnostics so only robust semantic regions are interpreted later.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
embedding_index = pd.read_csv(ROOT / "outputs/tables/article_embedding_index.csv")
embeddings = np.load(ROOT / "outputs/models/article_embeddings_main.npy")
work = embedding_index.merge(articles, on="article_id", how="left", suffixes=("", "_article")).sort_values("embedding_row")
print(work.shape, embeddings.shape)

## UMAP Runs

In [ ]:
import umap

UMAP_GRID = [
    {"n_neighbors": 10, "min_dist": 0.05, "metric": "cosine"},
    {"n_neighbors": 15, "min_dist": 0.10, "metric": "cosine"},
    {"n_neighbors": 30, "min_dist": 0.10, "metric": "cosine"},
    {"n_neighbors": 50, "min_dist": 0.20, "metric": "cosine"},
]
SEEDS = [1, 7, 42, 100, 2026]

umap_rows = []
for grid_id, params in enumerate(UMAP_GRID):
    for seed in SEEDS:
        run_id = f"umap_g{grid_id}_seed{seed}"
        out_path = ROOT / f"outputs/models/umap_runs/{run_id}.npy"
        if out_path.exists():
            xy = np.load(out_path)
        else:
            reducer = umap.UMAP(n_components=2, random_state=seed, **params)
            xy = reducer.fit_transform(embeddings)
            np.save(out_path, xy.astype(np.float32))
        umap_rows.append({"umap_run_id": run_id, "seed": seed, **params, "path": str(out_path.relative_to(ROOT))})

umap_index = pd.DataFrame(umap_rows)
write_csv(umap_index, ROOT / "outputs/tables/umap_run_index.csv")

main_umap = np.load(ROOT / "outputs/models/umap_runs/umap_g1_seed42.npy")
umap_main = work[["article_id", "year", "language", "title"]].copy()
umap_main["umap_x"] = main_umap[:, 0]
umap_main["umap_y"] = main_umap[:, 1]
write_csv(umap_main, ROOT / "outputs/tables/article_umap_main.csv")

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(umap_main["umap_x"], umap_main["umap_y"], c=umap_main["year"], cmap="viridis", s=16, alpha=0.75)
ax.set_title("HSR semantic map, colored by year")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
fig.colorbar(scatter, ax=ax, label="Year")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_semantic_map_by_year.png", dpi=220)
save_caption(ROOT, "fig_semantic_map_by_year.png", "Exploratory UMAP map colored by publication year; local neighborhoods and stable regions are interpreted, not point distances.")
plt.show()
display(umap_index.head())

## Cluster Solutions

In [ ]:
def label_summary(labels):
    labels = np.asarray(labels)
    non_noise = labels[labels != -1]
    return {
        "n_clusters": int(len(set(non_noise))),
        "n_noise": int((labels == -1).sum()),
        "noise_share": float((labels == -1).mean()),
    }

n_components = min(50, embeddings.shape[1], embeddings.shape[0] - 1)
pca = PCA(n_components=n_components, random_state=42)
pca_embeddings = pca.fit_transform(embeddings)
np.save(ROOT / "outputs/models/article_embeddings_pca50.npy", pca_embeddings.astype(np.float32))

solutions = {}
solution_meta = []

try:
    import hdbscan
    clusterer = hdbscan.HDBSCAN(min_cluster_size=18, min_samples=5, metric="euclidean", prediction_data=False)
    labels = clusterer.fit_predict(pca_embeddings)
    solutions["hdbscan_pca50"] = labels
    solution_meta.append({"cluster_solution_id": "hdbscan_pca50", "model_name": "HDBSCAN on PCA(50)", "clustering_params": "min_cluster_size=18; min_samples=5"})
except Exception as exc:
    print(f"External hdbscan package skipped: {exc}")
    try:
        from sklearn.cluster import HDBSCAN as SklearnHDBSCAN

        clusterer = SklearnHDBSCAN(min_cluster_size=18, min_samples=5, metric="euclidean")
        labels = clusterer.fit_predict(pca_embeddings)
        solutions["hdbscan_sklearn_pca50"] = labels
        solution_meta.append({"cluster_solution_id": "hdbscan_sklearn_pca50", "model_name": "scikit-learn HDBSCAN on PCA(50)", "clustering_params": "min_cluster_size=18; min_samples=5"})
    except Exception as exc2:
        print(f"scikit-learn HDBSCAN skipped: {exc2}")

nn = NearestNeighbors(n_neighbors=min(20, len(work) - 1), metric="cosine")
nn.fit(embeddings)
distances, neighbors = nn.kneighbors(embeddings)
G = nx.Graph()
G.add_nodes_from(range(len(work)))
for i, (row_dist, row_neighbors) in enumerate(zip(distances, neighbors)):
    for d, j in zip(row_dist[1:], row_neighbors[1:]):
        G.add_edge(i, int(j), weight=float(1 - d))
communities = nx.algorithms.community.louvain_communities(G, weight="weight", seed=42)
graph_labels = np.full(len(work), -1, dtype=int)
for community_id, nodes in enumerate(communities):
    for node in nodes:
        graph_labels[node] = community_id
solutions["knn_louvain"] = graph_labels
solution_meta.append({"cluster_solution_id": "knn_louvain", "model_name": "kNN graph Louvain", "clustering_params": "k=20; cosine"})

for n_clusters in [18, 24, 30]:
    km = KMeans(n_clusters=n_clusters, n_init="auto", random_state=42)
    labels = km.fit_predict(pca_embeddings)
    key = f"kmeans_pca50_k{n_clusters}"
    solutions[key] = labels
    solution_meta.append({"cluster_solution_id": key, "model_name": "KMeans on PCA(50)", "clustering_params": f"k={n_clusters}"})

meta_rows = []
for item in solution_meta:
    labels = solutions[item["cluster_solution_id"]]
    summary = label_summary(labels)
    try:
        sil = silhouette_score(pca_embeddings, labels, metric="euclidean") if summary["n_clusters"] > 1 else np.nan
    except Exception:
        sil = np.nan
    meta_rows.append({**item, **summary, "silhouette_highdim": sil, "embedding_model": embedding_index["embedding_model"].iloc[0]})

cluster_index = pd.DataFrame(meta_rows)

hdbscan_candidates = [key for key in ["hdbscan_pca50", "hdbscan_sklearn_pca50"] if key in solutions]
if hdbscan_candidates:
    candidate = hdbscan_candidates[0]
    candidate_row = cluster_index.loc[cluster_index["cluster_solution_id"].eq(candidate)].iloc[0]
    preferred = candidate if candidate_row["noise_share"] < 0.65 and candidate_row["n_clusters"] >= 5 else "knn_louvain"
else:
    preferred = "knn_louvain"
cluster_index["is_main_solution"] = cluster_index["cluster_solution_id"].eq(preferred)
write_csv(cluster_index, ROOT / "outputs/tables/cluster_solutions_index.csv")

assignments = work[["article_id", "year", "language", "title", "embedding_row"]].copy()
for key, labels in solutions.items():
    assignments[key] = labels
assignments["cluster_id_main"] = solutions[preferred]
assignments["cluster_solution_main"] = preferred
write_csv(assignments, ROOT / "outputs/tables/article_cluster_assignments_all_models.csv")

print(f"Main cluster solution: {preferred}")
display(cluster_index)

## Stability Atlas and Confound Checks

In [ ]:
main_labels = assignments["cluster_id_main"].to_numpy()
stability_rows = []
for key, labels in solutions.items():
    stability_rows.append(
        {
            "cluster_solution_id": key,
            "ari_to_main": adjusted_rand_score(main_labels, labels),
            "nmi_to_main": normalized_mutual_info_score(main_labels, labels),
            **label_summary(labels),
        }
    )
stability = pd.DataFrame(stability_rows).merge(cluster_index, on="cluster_solution_id", how="left", suffixes=("", "_index"))
write_csv(stability, ROOT / "outputs/tables/topic_stability_metrics.csv")

fig, ax = plt.subplots(figsize=(8, 4))
stability_plot = stability.set_index("cluster_solution_id")[["ari_to_main", "nmi_to_main", "noise_share"]]
sns.heatmap(stability_plot, annot=True, cmap="Blues", vmin=0, vmax=1, ax=ax)
ax.set_title("Model stability atlas")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_model_stability_atlas.png", dpi=220)
save_caption(ROOT, "fig_model_stability_atlas.png", "Comparison of clustering solutions against the selected main solution using ARI, NMI, and noise share.")
plt.show()

cluster_diag = (
    assignments.groupby("cluster_id_main")
    .agg(
        n_articles=("article_id", "nunique"),
        top_language=("language", lambda x: x.value_counts(dropna=False).index[0]),
        top_language_share=("language", lambda x: x.value_counts(normalize=True, dropna=False).iloc[0]),
        mean_text_length_words=("embedding_row", lambda rows: float(work.loc[rows.index, "text_length_words"].mean())),
        median_text_length_words=("embedding_row", lambda rows: float(work.loc[rows.index, "text_length_words"].median())),
    )
    .reset_index()
)
write_csv(cluster_diag, ROOT / "outputs/tables/language_cluster_diagnostics.csv")
write_csv(cluster_diag[["cluster_id_main", "n_articles", "mean_text_length_words", "median_text_length_words"]], ROOT / "outputs/tables/text_length_cluster_diagnostics.csv")

plot_df = umap_main.merge(assignments[["article_id", "cluster_id_main"]], on="article_id", how="left")
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=plot_df, x="umap_x", y="umap_y", hue="cluster_id_main", palette="tab20", s=16, alpha=0.8, legend=False, ax=ax)
ax.set_title("Main semantic regions")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_semantic_regions_main.png", dpi=220)
save_caption(ROOT, "fig_semantic_regions_main.png", "Main model-induced semantic regions overlaid on the exploratory UMAP map.")
plt.show()

display(stability)
display(cluster_diag.sort_values("top_language_share", ascending=False).head(15))